
# Amazon Reviews Scraping

In [13]:
# Import packages
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

In [14]:
# Header to set the requests as a browser requests
'''
These headers are used to mimic a browser request. This is a common 
practice in web scraping to prevent websites from blocking your requests.
'''
headers = {
    'authority': 'www.amazon.com',
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
    'accept-language': 'en-US,en;q=0.9,bn;q=0.8',
    'sec-ch-ua': '" Not A;Brand";v="99", "Chromium";v="102", "Google Chrome";v="102"',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36'
}

In [15]:
# url of The amazon Review page
reviews_url = 'https://www.amazon.in/Life-Pi-Booker-Prize-Winner/dp/0857865536/ref=sr_1_1?crid=UPD12SKPY2A7&dib=eyJ2IjoiMSJ9'

In [16]:
# number of pages to scrape.
len_page = 4

### Functions

In [17]:
# This function retrieves the HTML data from multiple pages of 
# Amazon reviews.

# Extra Data as Html object from amazon Review page
def reviewsHtml(url, len_page):
    
    # Empty List define to store all pages html data
    soups = []
    
    # Loop for gather all 3000 reviews from 300 pages via range
    for page_no in range(1, len_page + 1):
        
        # parameter set as page no to the requests body
        params = {
            'ie': 'UTF8',
            'reviewerType': 'all_reviews',
            'filterByStar': 'critical',
            'pageNumber': page_no,
        }
        
        # Request make for each page
        response = requests.get(url, headers=headers)
        
        # Save Html object by using BeautifulSoup4 and lxml parser
        soup = BeautifulSoup(response.text, 'lxml')
        
        # Add single Html page data in master soups list
        soups.append(soup)
        
    return soups

In [18]:
# This function extracts relevant information from the HTML data of each review.
# Grab Reviews name, description, date, stars, title from HTML
def getReviews(html_data):

    # Create Empty list to Hold all data
    data_dicts = []
    
    # Select all Reviews BOX html using css selector
    boxes = html_data.select('div[data-hook="review"]')
    
    # Iterate all Reviews BOX 
    for box in boxes:
        
        # Select Name using css selector and cleaning text using strip()
        # If Value is empty define value with 'N/A' for all.
        try:
            name = box.select_one('[class="a-profile-name"]').text.strip()
        except Exception as e:
            name = 'N/A'

        try:
            stars = box.select_one('[data-hook="review-star-rating"]').text.strip().split(' out')[0]
        except Exception as e:
            stars = 'N/A'   

        try:
            title = box.select_one('[data-hook="review-title"]').text.strip()
        except Exception as e:
            title = 'N/A'

        try:
            # Convert date str to dd/mm/yyy format
            datetime_str = box.select_one('[data-hook="review-date"]').text.strip().split(' on ')[-1]
            date = datetime.strptime(datetime_str, '%B %d, %Y').strftime("%d/%m/%Y")
        except Exception as e:
            date = 'N/A'

        try:
            description = box.select_one('[data-hook="review-body"]').text.strip()
        except Exception as e:
            description = 'N/A'

        # create Dictionary with al review data 
        data_dict = {
            'Name' : name,
            'Stars' : stars,
            'Title' : title,
            'Date' : date,
            'Description' : description
        }

        # Add Dictionary in master empty List
        data_dicts.append(data_dict)
    
    return data_dicts

### Data Process

In [19]:
# Grab all HTML
html_datas = reviewsHtml(reviews_url, len_page)

In [20]:
# Empty List to Hold all reviews data
reviews = []

In [21]:
# Iterate all Html page 
for html_data in html_datas:
    
    # Grab review data
    review = getReviews(html_data)
    
    # add review data in reviews empty list
    reviews += review

In [22]:
# Create a dataframe with reviews Data
df_reviews = pd.DataFrame(reviews)

In [23]:
df_reviews

,Name,Stars,Title,Date,Description
0,₹ockey,5,N/A,N/A,N/A
1,Minakshi Sharma,5,N/A,N/A,N/A
2,Singh,4,N/A,N/A,N/A
3,Priyanka Yadav,5,N/A,N/A,N/A
4,Siya Rai,5,N/A,N/A,N/A
5,Nilanjana,5,N/A,N/A,N/A
6,Rohit Kumar Singh,4,N/A,N/A,N/A
7,Nadasha,5,N/A,N/A,N/A
8,x,5,N/A,N/A,N/A
9,Curious Puppy,5,N/A,N/A,N/A


In [12]:
# Save data
df_reviews.to_csv('reviews.csv', index=False)